In [3]:
import pandas as pd 
import requests

In [4]:
from pytrends.request import TrendReq
from pytrends.exceptions import TooManyRequestsError
import time

# Connect to Google Trends
pytrends = TrendReq()

# Search keywords
keywords = ["Bitcoin", "Ethereum"]

# Date range (90-day intervals)
dates = pd.date_range("2021-06-01", "2026-06-01", freq="90D")

# Store data
all_data = []

# Download Google Trends data
for i in range(len(dates) - 1):

    start = dates[i].strftime("%Y-%m-%d")
    end = dates[i + 1].strftime("%Y-%m-%d")

    print(f"Downloading {start} to {end}")

    try:
        pytrends.build_payload(keywords, timeframe=f"{start} {end}")

        data = pytrends.interest_over_time()

        if not data.empty:
            data = data.drop(columns=["isPartial"], errors="ignore")
            all_data.append(data)

        time.sleep(20)

    except TooManyRequestsError:
        print("Too many requests. Waiting 2 minutes...")
        time.sleep(120)

# Combine all data
trends_data = pd.concat(all_data)

# Remove duplicate dates
trends_data = trends_data[~trends_data.index.duplicated()]

# Save CSV
trends_data.to_csv("../data/raw/google_trends.csv")

print("Google Trends data saved successfully!")

Google Trends data saved successfully!


In [3]:
import pandas as pd
import requests

url = "https://api.alternative.me/fng/?limit=0&format=json"

response = requests.get(url)
data = response.json()["data"]

fear_greed = pd.DataFrame(data)

fear_greed = fear_greed[["timestamp", "value", "value_classification"]]

fear_greed.columns = ["Date", "Fear_Greed_Index", "Classification"]

# Convert timestamp text to number first
fear_greed["Date"] = pd.to_numeric(fear_greed["Date"])
fear_greed["Date"] = pd.to_datetime(fear_greed["Date"], unit="s")

# Convert index value to number
fear_greed["Fear_Greed_Index"] = pd.to_numeric(fear_greed["Fear_Greed_Index"])

# Keep required dates
fear_greed = fear_greed[
    (fear_greed["Date"] >= "2021-06-01") &
    (fear_greed["Date"] <= "2026-06-01")
]

fear_greed = fear_greed.sort_values("Date")

fear_greed.to_csv("../data/raw/fear_greed.csv", index=False)

print("Start Date:", fear_greed["Date"].min())
print("End Date:", fear_greed["Date"].max())
print("Total Days:", len(fear_greed))

Start Date: 2021-06-01 00:00:00
End Date: 2026-06-01 00:00:00
Total Days: 1826
